This project is an autonomous conversational agent built using LangGraph and OpenAI, featuring a multi-thread Gradio web interface. It utilizes the Model Context Protocol (MCP) to integrate a persistent LibSQL knowledge graph, empowering the AI to proactively save and recall personal facts across entirely different chat sessions. Finally, the asynchronous architecture is secured by a custom Pydantic-driven guardrail node that evaluates all user inputs for safety prior to standard LLM processing.

In [ ]:
#import sqlite3
import aiosqlite
import uuid
import os
import gradio as gr
from typing import Annotated
from typing_extensions import TypedDict
from dotenv import load_dotenv
from datetime import datetime

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
#from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langgraph.prebuilt import ToolNode # Brings tools into the graph

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_mcp_adapters.client import MultiServerMCPClient # Manages MCP connections
from pydantic import BaseModel, Field

In [ ]:
load_dotenv(override=True)

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
# Define the MCP Servers (Brave for Search, LibSQL for Memory)
brave_env = {"BRAVE_API_KEY": os.environ.get("BRAVE_API_KEY", "")}

mcp_client = MultiServerMCPClient({
    "brave": {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-brave-search"],
        "env": brave_env,
        "transport": "stdio"
    },
    "memory": {
        "command": "npx",
        "args": ["-y", "mcp-memory-libsql"],
        "env": {"LIBSQL_URL": "file:./mcp_memory.db"}, # Saves graph memory locally
        "transport": "stdio"
    }
})

In [ ]:
mcp_tools = await mcp_client.get_tools()

In [ ]:
llm_with_tools = llm.bind_tools(mcp_tools)

In [ ]:
# Define Graph State and Async Nodes

class State(TypedDict):
    messages: Annotated[list, add_messages] 
    is_safe: bool                           
    title: str                              

class InputGuardrail(BaseModel):
    is_safe: bool = Field(description="True if the user input is safe. False if it asks for malicious code, system commands, or inappropriate content.")

# Nodes are made async to handle MCP communication
async def guardrail_node(state: State):
    user_message = state["messages"][-1].content
    analyzer = llm.with_structured_output(InputGuardrail)
    prompt = f"You are a security guardrail. Is this safe? Input: {user_message}"
    result = await analyzer.ainvoke(prompt)
    return {"is_safe": result.is_safe}

async def chatbot_node(state: State):
    messages = state["messages"]
    if not isinstance(messages[0], SystemMessage):
        current_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        system_prompt = SystemMessage(
            content=f"""You are an autonomous AI with persistent memory. 
            The current date and time is {current_date}.

            CRITICAL INSTRUCTIONS FOR MEMORY USAGE:
            You have tools to interact with a knowledge graph (LibSQL Memory). 
            1. SAVING: Whenever the user tells you their name, preferences, or personal facts, you MUST immediately use your tools to save this to the knowledge graph. Do not just reply; save it first!
            2. RECALLING: If the user asks you to remember something, or asks about themselves, you MUST use your tools to query the knowledge graph before answering. 

            Do not rely on your short-term conversation history for facts. Write them down!"""
        )
        messages = [system_prompt] + messages
        
    response = await llm_with_tools.ainvoke(messages)
    return {"messages": [response]}

def refusal_node(state: State):
    rejection = AIMessage(content="I cannot fulfill this request as it violates my safety guidelines.")
    return {"messages": [rejection]}

async def generate_title_node(state: State):
    first_user_message = [m.content for m in state["messages"] if isinstance(m, HumanMessage)][0]
    prompt = f"Summarize this user query in 3 to 4 words. Query: {first_user_message}"
    title_response = await llm.ainvoke(prompt)
    return {"title": title_response.content}

Build the Graph Architecture

In [ ]:
# Create a LangGraph Node specifically for executing tools
tool_node = ToolNode(mcp_tools)

In [ ]:
def route_after_guardrail(state: State):
    return "chatbot" if state.get("is_safe") else "refusal"

def route_after_chatbot(state: State):
    """Dynamically routes based on whether the AI decided to use a tool."""
    last_msg = state["messages"][-1]
    
    # If the LLM returned tool calls, go to the tool execution node
    if hasattr(last_msg, "tool_calls") and len(last_msg.tool_calls) > 0:
        return "tools"
    
    # If no tools were called, wrap up the cycle
    if not state.get("title"):
        return "generate_title"
    return END

In [ ]:
# Assemble the Graph
graph_builder = StateGraph(State)

graph_builder.add_node("guardrail", guardrail_node)
graph_builder.add_node("chatbot", chatbot_node)
graph_builder.add_node("tools", tool_node) # Add the new tool node
graph_builder.add_node("refusal", refusal_node)
graph_builder.add_node("generate_title", generate_title_node)

graph_builder.add_edge(START, "guardrail")
graph_builder.add_conditional_edges("guardrail", route_after_guardrail, {"chatbot": "chatbot", "refusal": "refusal"})
graph_builder.add_edge("refusal", END)

# Add conditional routing for the tools
graph_builder.add_conditional_edges("chatbot", route_after_chatbot, {"tools": "tools", "generate_title": "generate_title", END: END})

# Once the tool finishes executing, loop back to the chatbot to read the results
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("generate_title", END)

#conn = sqlite3.connect("my_chat_history.db", check_same_thread=False)
#memory = SqliteSaver(conn)
conn = await aiosqlite.connect("my_chat_history.db")
memory = AsyncSqliteSaver(conn)
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# Maintain an ordered list of Thread IDs and a mapping to their display titles
thread_ids = [str(uuid.uuid4())]
thread_titles = {thread_ids[0]: "New Chat"}

In [ ]:
def get_gradio_history(thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}
    state = graph.get_state(config)
    if not state or "messages" not in state.values:
        return []
    
    gradio_messages = []
    for msg in state.values["messages"]:
        role = "user" if isinstance(msg, HumanMessage) else "assistant"
        # filter out raw 'ToolMessage' outputs so they don't clutter the UI
        if type(msg) in [HumanMessage, AIMessage] and msg.content:
            gradio_messages.append({"role": role, "content": msg.content})
            
    return gradio_messages

def get_chat_list(): return [[thread_titles[tid]] for tid in thread_ids]

def start_new_chat():
    new_tid = str(uuid.uuid4())
    thread_ids.append(new_tid)
    thread_titles[new_tid] = "New Chat"
    return get_chat_list(), [], new_tid

def switch_chat(evt: gr.SelectData):
    selected_tid = thread_ids[evt.index[0]]
    return get_gradio_history(selected_tid), selected_tid

# Gradio submission must be async to run our async graph
async def submit_message(user_input, current_thread_id):
    if not user_input.strip():
        return gr.update(), gr.update(), gr.update() 

    config = {"configurable": {"thread_id": current_thread_id}}
    
    # Use ainvoke() instead of invoke()
    await graph.ainvoke(
        {"messages": [{"role": "user", "content": user_input}]}, 
        config=config
    )
    
    state = graph.get_state(config)
    thread_titles[current_thread_id] = state.values.get("title", "New Chat")
    
    return get_gradio_history(current_thread_id), "", get_chat_list()

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    current_thread = gr.State(thread_ids[0])
    gr.Markdown("## Autonomous Agent: Web Search & Memory Edition")
    
    with gr.Row():
        with gr.Column(scale=1):
            new_chat_btn = gr.Button("➕ New Chat", variant="primary")
            chat_list_ui = gr.Dataframe(headers=["Conversations"], value=get_chat_list(), interactive=False, row_count=(1, "dynamic"), col_count=(1, "fixed"))
            
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=500, type="messages")
            msg_input = gr.Textbox(placeholder="Type your message...", show_label=False)
            
    msg_input.submit(submit_message, inputs=[msg_input, current_thread], outputs=[chatbot, msg_input, chat_list_ui])
    new_chat_btn.click(start_new_chat, inputs=[], outputs=[chat_list_ui, chatbot, current_thread])
    chat_list_ui.select(switch_chat, inputs=[], outputs=[chatbot, current_thread])


In [ ]:
demo.launch()
